# TakeMeter — RoBERTa Fine-Tuning (3-class)

**Runtime → Change runtime type → T4 GPU** before running.

Changes from the bake-off notebook:
- Informational merged into **Evaluative** → 3 classes: Analytical / Evaluative / Reactive
- Single model: `roberta-base`
- LR grid: `[2e-5, 3e-5, 5e-5]` (roberta was still climbing at 3e-5)
- `fp16=True` for faster training

Workflow: GPU check → install → upload data → HF login → CV → final train → push

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

## 3. Upload dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # select dataset.csv

import pandas as pd
df_check = pd.read_csv("dataset.csv")
print(f"Rows: {len(df_check)}")
print("Original label counts:")
print(df_check["label"].value_counts())

## 4. Log in to Hugging Face

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Config & imports

In [ ]:
import json
import os
import random
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight

import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── 3-class label scheme (Informational merged into Evaluative) ───────────────
LABEL_MERGE = {"Informational": "Evaluative"}  # remap before encoding
LABEL2ID    = {"Analytical": 0, "Evaluative": 1, "Reactive": 2}
ID2LABEL    = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS  = len(LABEL2ID)

MODEL_ID   = "roberta-base"
LR_GRID    = [2e-5, 3e-5, 5e-5]
N_FOLDS    = 5
SEED       = 42
MAX_LEN    = 256
BATCH_SIZE = 16
MAX_EPOCHS = 10
PATIENCE   = 3

print("Config ready. Labels:", LABEL2ID)

## 6. Data loading & label merging

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def build_input_text(row):
    tag = "[POST]" if row["type"] == "post" else "[COMMENT]"
    return f"{tag} {row['post_title']}\n{row['text']}"

set_seed(SEED)

df = pd.read_csv("dataset.csv")
df = df[df["label"].notna() & (df["label"].str.strip() != "")].copy()

# Merge Informational → Evaluative
df["label"] = df["label"].replace(LABEL_MERGE)

df["input_text"] = df.apply(build_input_text, axis=1)
df["label_id"]   = df["label"].map(LABEL2ID)
df = df.reset_index(drop=True)

cv_df, holdout_df = train_test_split(
    df, test_size=40, stratify=df["label_id"], random_state=SEED
)
cv_df      = cv_df.reset_index(drop=True)
holdout_df = holdout_df.reset_index(drop=True)

print(f"CV pool:      {len(cv_df)} examples")
print(f"Hold-out set: {len(holdout_df)} examples")
print("\nCV label distribution (post-merge):")
print(cv_df["label"].value_counts())

## 7. Training utilities

In [ ]:
def make_hf_dataset(texts, label_ids, tokenizer):
    ds = Dataset.from_dict({"input_text": texts, "labels": label_ids})
    def tokenize(batch):
        return tokenizer(batch["input_text"], truncation=True,
                         max_length=MAX_LEN, padding="max_length")
    ds = ds.map(tokenize, batched=True, remove_columns=["input_text"])
    ds.set_format("torch")
    return ds


class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(outputs.logits.device)
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}


def train_fold(lr, fold_idx, train_df, val_df, class_weights):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    train_ds  = make_hf_dataset(train_df["input_text"].tolist(), train_df["label_id"].tolist(), tokenizer)
    val_ds    = make_hf_dataset(val_df["input_text"].tolist(),   val_df["label_id"].tolist(),   tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=NUM_LABELS,
        id2label=ID2LABEL, label2id=LABEL2ID,
    )

    warmup_steps = max(1, int(0.1 * (len(train_ds) // BATCH_SIZE) * MAX_EPOCHS))

    training_args = TrainingArguments(
        output_dir=f"/content/ckpts/fold{fold_idx}",
        num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=lr,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        seed=SEED,
        report_to="none",
        logging_steps=50,
        fp16=True,
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    trainer.train()

    pred_out = trainer.predict(val_ds)
    preds    = np.argmax(pred_out.predictions, axis=-1)
    true     = pred_out.label_ids
    macro_f1 = f1_score(true, preds, average="macro", zero_division=0)

    print(f"  Fold {fold_idx+1} macro-F1: {macro_f1:.4f}")
    print(classification_report(true, preds, target_names=list(LABEL2ID.keys()), zero_division=0))

    per_class = f1_score(true, preds, average=None, zero_division=0)
    return {
        "fold": fold_idx,
        "macro_f1": macro_f1,
        "per_class_f1": {k: float(per_class[v]) for k, v in LABEL2ID.items()},
        "confusion_matrix": confusion_matrix(true, preds).tolist(),
    }


def run_cv(lr, data_df):
    set_seed(SEED)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    class_weights = torch.tensor(
        compute_class_weight("balanced", classes=np.arange(NUM_LABELS), y=data_df["label_id"].values),
        dtype=torch.float32,
    )

    fold_results = []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(data_df, data_df["label_id"])):
        print(f"\n{'='*60}")
        print(f"  roberta-base  |  lr={lr:.0e}  |  fold {fold_idx+1}/{N_FOLDS}")
        print(f"{'='*60}")
        result = train_fold(
            lr, fold_idx,
            data_df.iloc[train_idx].reset_index(drop=True),
            data_df.iloc[val_idx].reset_index(drop=True),
            class_weights,
        )
        fold_results.append(result)

    macro_f1s = [r["macro_f1"] for r in fold_results]
    mean, std = float(np.mean(macro_f1s)), float(np.std(macro_f1s))
    print(f"\n>>> roberta-base  lr={lr:.0e}  macro-F1 = {mean:.4f} ± {std:.4f}")
    return {"model": "roberta-base", "lr": lr, "mean_macro_f1": mean, "std_macro_f1": std, "folds": fold_results}


print("Utilities ready.")

## 8. Cross-validation — roberta-base

3 learning rates × 5 folds = 15 training runs (~20–30 min on T4 with fp16).

In [ ]:
cv_results = []

for lr in LR_GRID:
    result = run_cv(lr, cv_df)
    cv_results.append(result)
    with open("cv_results_roberta_3class.json", "w") as f:
        json.dump(cv_results, f, indent=2)

print("\n── roberta-base 3-class leaderboard ──")
for r in sorted(cv_results, key=lambda x: x["mean_macro_f1"], reverse=True):
    print(f"  lr={r['lr']:.0e}  macro-F1 = {r['mean_macro_f1']:.4f} ± {r['std_macro_f1']:.4f}")

## 9. Pick best LR

In [ ]:
best = max(cv_results, key=lambda x: x["mean_macro_f1"])
BEST_LR = best["lr"]
HF_REPO = "your-username/takemeter"  # ← replace with your HF username

print(f"Best LR: {BEST_LR:.0e}  (macro-F1 = {best['mean_macro_f1']:.4f} ± {best['std_macro_f1']:.4f})")
print(f"Will push to: {HF_REPO}")

## 10. Final training — all CV data

In [ ]:
set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

class_weights = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(NUM_LABELS), y=cv_df["label_id"].values),
    dtype=torch.float32,
)

train_ds   = make_hf_dataset(cv_df["input_text"].tolist(),      cv_df["label_id"].tolist(),      tokenizer)
holdout_ds = make_hf_dataset(holdout_df["input_text"].tolist(), holdout_df["label_id"].tolist(), tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_LABELS,
    id2label=ID2LABEL, label2id=LABEL2ID,
)

warmup_steps = max(1, int(0.1 * (len(train_ds) // BATCH_SIZE) * MAX_EPOCHS))

final_args = TrainingArguments(
    output_dir="/content/takemeter-final",
    num_train_epochs=MAX_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=BEST_LR,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    seed=SEED,
    report_to="none",
    logging_steps=50,
    fp16=True,
    push_to_hub=True,
    hub_model_id=HF_REPO,
)

final_trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model, args=final_args,
    train_dataset=train_ds, eval_dataset=holdout_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    tokenizer=tokenizer,
)

final_trainer.train()
print("Training complete.")

## 11. Final evaluation on hold-out (run once)

In [ ]:
pred_out    = final_trainer.predict(holdout_ds)
final_preds = np.argmax(pred_out.predictions, axis=-1)
final_true  = pred_out.label_ids
final_macro = f1_score(final_true, final_preds, average="macro", zero_division=0)

print(f"Hold-out macro-F1: {final_macro:.4f}")
print()
print(classification_report(final_true, final_preds, target_names=list(LABEL2ID.keys()), zero_division=0))
print("Confusion matrix (rows=true, cols=pred):")
cm = confusion_matrix(final_true, final_preds)
cm_df = pd.DataFrame(cm, index=list(LABEL2ID.keys()), columns=list(LABEL2ID.keys()))
print(cm_df)

## 12. Push to Hugging Face Hub

In [ ]:
cv_summary = f"{best['mean_macro_f1']:.4f} ± {best['std_macro_f1']:.4f} (lr={BEST_LR:.0e})"

model_card = f"""---
language: en
tags:
- text-classification
- anime
- discourse
pipeline_tag: text-classification
---

# TakeMeter — Anime Discourse Classifier

Fine-tuned **roberta-base** for 3-class discourse mode classification of anime community posts.

## Labels
| ID | Label | Description |
|---|---|---|
| 0 | Analytical | Argues an interpretation with text evidence |
| 1 | Evaluative | Asserts a judgment, ranking, or supplies factual/lore information |
| 2 | Reactive | Expresses emotion or hype, no substantive claim |

## Training
- **Data:** labeled posts/comments from r/attackontitan, r/HunterXHunter, r/FullmetalAlchemist, r/evangelion
- **CV macro-F1:** {cv_summary} (5-fold stratified)
- **Hold-out macro-F1:** {final_macro:.4f}
- **Class weighting:** inverse-frequency

## Usage
```python
from transformers import pipeline
classifier = pipeline(\"text-classification\", model=\"{HF_REPO}\")
classifier(\"[POST] Was the ending good?\\nEren's arc was the most compelling in the series.\")
# → {{\"label\": \"Analytical\", \"score\": 0.87}}
```
"""

with open("/content/takemeter-final/README.md", "w") as f:
    f.write(model_card)

final_trainer.push_to_hub(commit_message=f"Final model — hold-out macro-F1 {final_macro:.4f}")
print(f"Pushed to https://huggingface.co/{HF_REPO}")

## 13. Download CV results

In [ ]:
from google.colab import files
files.download("cv_results_roberta_3class.json")